In [ ]:
import torch
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms
import matplotlib.pyplot as plt

device = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"

In [ ]:
tfm = transforms.ToTensor()  # PIL image -> tensor, pixels scaled to [0,1]
train_ds = datasets.MNIST("data", train=True, download=True, transform=tfm)
test_ds  = datasets.MNIST("data", train=False, download=True, transform=tfm)

train_small  = Subset(train_ds, range(10_000))  # small dataset, as assigned
train_loader = DataLoader(train_small, batch_size=128, shuffle=True)
test_loader  = DataLoader(test_ds, batch_size=128)

Noise

In [ ]:
def add_noise(x, sigma=0.3):
    return (x + sigma * torch.randn_like(x)).clamp(0, 1)

Model

In [ ]:
class AE(nn.Module):
    def __init__(self, latent=32):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Flatten(), nn.Linear(784, 128), nn.ReLU(), nn.Linear(128, latent))
        self.decoder = nn.Sequential(
            nn.Linear(latent, 128), nn.ReLU(), nn.Linear(128, 784),
            nn.Sigmoid(), nn.Unflatten(1, (1, 28, 28)))

    def forward(self, x):
        return self.decoder(self.encoder(x))

Training

In [ ]:
model = AE().to(device)
opt = torch.optim.Adam(model.parameters(), lr=1e-3)
loss_fn = nn.MSELoss()

for epoch in range(10):
    total = 0
    for x, _ in train_loader:          # labels (_) are unused: unsupervised
        x = x.to(device)
        noisy = add_noise(x)           # fresh noise every batch
        loss = loss_fn(model(noisy), x)  # compare output to the CLEAN image
        opt.zero_grad(); loss.backward(); opt.step()
        total += loss.item() * x.size(0)
    print(f"epoch {epoch+1}: loss {total/len(train_small):.4f}")

Plotting

In [ ]:
model.eval()
x, _ = next(iter(test_loader))
x = x[:8].to(device)
noisy = add_noise(x)
with torch.no_grad():
    recon = model(noisy)

fig, axes = plt.subplots(3, 8, figsize=(12, 5))
for row, imgs, title in zip(axes, [x, noisy, recon], ["clean", "noisy", "denoised"]):
    for ax, img in zip(row, imgs.cpu()):
        ax.imshow(img.squeeze(), cmap="gray"); ax.axis("off")
    row[0].set_title(title, loc="left")
plt.show()